# FASTT — rebuilding the features from the audio (level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/fastt/blob/main/notebooks/extract.ipynb)

The selectors learn from three things per corpus and expert: the expert's **transcripts**, their **word-error
counters** against the reference, and the **mean of its final-layer encoder states** per clip. Levels 1 and 2 read
them from the published dataset; this notebook rebuilds them from the audio: every expert of every corpus's trio
decodes the corpus in full.

**Where it writes.** A local folder, `$FASTT_CACHE/rebuilt` (by default `~/.cache/fastt/rebuilt`). Then run
`main_<corpus>` at level 3 **on the same machine**: it reads the features from that folder, the published searches
from the public dataset, and refits every fold. Nothing is uploaded.

**Three toolkits.** The experts come from three toolkits that do not install into one environment: **A**
🤗 Transformers (wav2vec2, data2vec, Whisper), **B** NeMo (Parakeet-TDT, Canary-Qwen), **C** ESPnet. Set `GROUP`,
run the notebook, and repeat in a fresh runtime for each group the trios need (LibriSpeech: A and B; the other four
corpora: A and C). An expert already complete in the folder is skipped.

**Cost.** Each (corpus, expert) is one pass of the model over the corpus: minutes to an hour on a single GPU. The
audio is read from the published `base` configuration of each corpus (16 kHz, the clips the paper uses).

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("fastt") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fastt[paper] @ git+https://github.com/huseyin-karaca/fastt.git@main"], check=True)

In [ ]:
GROUP = "A"                                                  # "A", "B" or "C" (see above)
CORPORA = ("librispeech", "voxpopuli", "ami_sdm", "earnings22", "peoples_speech")   # or a subset

import subprocess, sys
from fastt.rebuild import INSTALL
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *INSTALL[GROUP]], check=True)

## 1. What to rebuild

Every expert of each corpus's trio (`fastt.configs.MAIN[corpus].members`) that belongs to `GROUP` and is not yet
complete in the folder.

In [ ]:
from fastt.rebuild import compare_with_published, rebuild, rebuild_plan

plan = rebuild_plan(CORPORA, group=GROUP)
plan

## 2. The pass

Per corpus the audio is loaded once; then per expert a short smoke test (shapes and one transcript) and the full
pass, greedy decoding, written to the folder.

In [ ]:
log = rebuild(plan)
log

## 3. Check against the published features

The same checkpoint, decoder and normaliser give the same transcripts up to the GPU's numerics: nearly every
normalised transcript is identical, and each expert's corpus WER matches the published one to about the third
decimal.

In [ ]:
compare_with_published(log)